In [0]:
CREATE OR REPLACE VIEW ademianczuk.supply_chain_360.silver_receipt_line_enriched AS
SELECT
  rl.receipt_id,
  rl.po_id,
  rl.dc_id,
  dc.region,

  rl.supplier_id,

  rl.received_date,
  rl.promised_date,
  datediff(rl.received_date, rl.promised_date) AS days_late,
  CASE WHEN datediff(rl.received_date, rl.promised_date) <= 0 THEN 1 ELSE 0 END AS on_time_flag,

  rl.sku_id,
  sku.brand,
  sku.unit_cost,
  rl.category,

  rl.ordered_units,
  rl.received_units,
  CASE WHEN rl.received_units >= rl.ordered_units THEN 1 ELSE 0 END AS in_full_flag,

  greatest(rl.ordered_units - rl.received_units, 0) AS short_units,

  rl.defect_units,
  (rl.defect_units / NULLIF(rl.received_units, 0)) AS defect_rate,

  CASE WHEN datediff(rl.received_date, rl.promised_date) <= 0
            AND rl.received_units >= rl.ordered_units
       THEN 1 ELSE 0 END AS otif_flag,

  rl.shock_flag,
  rl.supplier_id AS supplier_key,
  rl.dc_id AS dc_key,
  rl.sku_id AS sku_key
FROM ademianczuk.supply_chain_360.fact_receipt_line rl
LEFT JOIN ademianczuk.supply_chain_360.dim_dc dc        ON dc.dc_id = rl.dc_id
LEFT JOIN ademianczuk.supply_chain_360.dim_supplier sup ON sup.supplier_id = rl.supplier_id
LEFT JOIN ademianczuk.supply_chain_360.dim_sku sku      ON sku.sku_id = rl.sku_id;

In [0]:
CREATE OR REPLACE VIEW ademianczuk.supply_chain_360.silver_shipment_enriched AS
SELECT
  s.shipment_id,
  s.dc_id,
  dc.region as `dc_region`,
  s.store_id,
  s.region as `store_region`,
  s.carrier,
  s.mode,
  s.ship_date,
  s.delivery_date,
  datediff(s.delivery_date, s.ship_date) AS transit_days,

  /* SLA heuristic: on-time if delivered within 2 days of ship_date */
  CASE WHEN datediff(s.delivery_date, s.ship_date) <= 2 THEN 1 ELSE 0 END AS on_time_flag,
  CASE WHEN datediff(s.delivery_date, s.ship_date) > 2 THEN 1 ELSE 0 END AS late_flag,

  s.cost,
  s.shock_flag
FROM ademianczuk.supply_chain_360.fact_shipment s
LEFT JOIN ademianczuk.supply_chain_360.dim_dc dc     ON dc.dc_id = s.dc_id
LEFT JOIN ademianczuk.supply_chain_360.dim_store st  ON st.store_id = s.store_id;

In [0]:
CREATE OR REPLACE VIEW ademianczuk.supply_chain_360.silver_inventory_demand_daily AS
WITH dc_sales AS (
  SELECT
    date,
    dc_id,
    sku_id,
    SUM(units_sold) AS units_sold_dc
  FROM ademianczuk.supply_chain_360.fact_sales_daily
  GROUP BY date, dc_id, sku_id
),
dc_sales_7d AS (
  SELECT
    d1.date,
    d1.dc_id,
    d1.sku_id,
    /* 7-day trailing average daily demand */
    AVG(d2.units_sold_dc) AS avg_daily_demand_7d
  FROM dc_sales d1
  JOIN dc_sales d2
    ON d2.dc_id = d1.dc_id
   AND d2.sku_id = d1.sku_id
   AND d2.date BETWEEN date_sub(d1.date, 6) AND d1.date
  GROUP BY d1.date, d1.dc_id, d1.sku_id
)
SELECT
  i.date,
  i.dc_id,
  dc.region as `dc_region`,
  i.sku_id,
  sku.category,
  i.region,
  i.on_hand_units,
  i.on_order_units,
  COALESCE(s7.avg_daily_demand_7d, 0) AS avg_daily_demand_7d,
  (i.on_hand_units / NULLIF(s7.avg_daily_demand_7d, 0)) AS days_of_supply,
  i.shock_flag
FROM ademianczuk.supply_chain_360.fact_inventory_daily i
LEFT JOIN dc_sales_7d s7
  ON s7.date = i.date AND s7.dc_id = i.dc_id AND s7.sku_id = i.sku_id
LEFT JOIN ademianczuk.supply_chain_360.dim_dc dc
  ON dc.dc_id = i.dc_id
LEFT JOIN ademianczuk.supply_chain_360.dim_sku sku
  ON sku.sku_id = i.sku_id;

In [0]:
CREATE OR REPLACE VIEW ademianczuk.supply_chain_360.silver_sales_enriched AS
SELECT
  s.date,
  s.store_id,
  st.region as `store_region`,
  s.dc_id,
  dc.region as `dc_region`,
  s.sku_id,
  sku.category,
  s.units_sold,
  s.price,
  (s.units_sold * s.price) AS sales_amount,
  s.promo_flag,
  s.region,
  s.shock_flag
FROM ademianczuk.supply_chain_360.fact_sales_daily s
LEFT JOIN ademianczuk.supply_chain_360.dim_store st ON st.store_id = s.store_id
LEFT JOIN ademianczuk.supply_chain_360.dim_dc dc    ON dc.dc_id = s.dc_id
LEFT JOIN ademianczuk.supply_chain_360.dim_sku sku  ON sku.sku_id = s.sku_id;